# DDM Fixed Bound Analysis
## nxx1, seed=42, gain=1.0
Model: v ~ 1 + coherence (fixed threshold)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
%matplotlib inline

In [ ]:
import xarray as xr
idata = xr.open_datatree('/Users/aliciasmacbookair/Desktop/rnn_hssm_output/ddm_fixed_nxx1_s42_g1.0')
print(idata)

## Model Summary

In [ ]:
# Parameter summary
posterior = idata.posterior
params = ['v_Intercept', 'v_coherence', 'a', 'z', 't', 'p_outlier']
summary_data = []
for param in params:
    data = posterior[param].values.flatten()
    summary_data.append({
        'parameter': param,
        'mean': data.mean(),
        'sd': data.std(),
        'hdi_3%': np.percentile(data, 3),
        'hdi_97%': np.percentile(data, 97),
    })
pd.DataFrame(summary_data).set_index('parameter').round(3)

## Traces

In [ ]:
# Plot traces manually
posterior = idata.posterior
params = ['v_Intercept', 'v_coherence', 'a', 'z', 't', 'p_outlier']
fig, axes = plt.subplots(len(params), 2, figsize=(12, 3*len(params)))
for i, param in enumerate(params):
    data = posterior[param].values  # shape: (chains, draws)
    # KDE plot
    for chain in data:
        axes[i, 0].hist(chain, bins=30, alpha=0.5, density=True)
    axes[i, 0].set_title(param)
    axes[i, 0].set_xlabel('value')
    # Trace plot
    for chain in data:
        axes[i, 1].plot(chain, alpha=0.7)
    axes[i, 1].set_title(f'{param} trace')
plt.tight_layout()
plt.show()

## Posterior Distributions

In [ ]:
# Plot posterior distributions
posterior = idata.posterior
params = ['v_Intercept', 'v_coherence', 'a', 'z', 't', 'p_outlier']
fig, axes = plt.subplots(1, len(params), figsize=(15, 3))
for i, param in enumerate(params):
    data = posterior[param].values.flatten()
    axes[i].hist(data, bins=50, density=True, alpha=0.7, color='steelblue')
    axes[i].axvline(data.mean(), color='red', linestyle='--', label=f'mean={data.mean():.3f}')
    axes[i].set_title(param)
    axes[i].legend(fontsize=8)
plt.tight_layout()
plt.show()

## Effect of Coherence on Drift Rate

In [ ]:
import xarray as xr
posterior = idata.posterior
v_intercept = float(posterior['v_Intercept'].mean())
v_coherence = float(posterior['v_coherence'].mean())

coherence_vals = np.linspace(0, 0.15, 100)
drift_rate = v_intercept + coherence_vals * v_coherence

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(coherence_vals, drift_rate, color='blue')
ax.set_xlabel('Coherence (|coh|)')
ax.set_ylabel('Drift rate (v)')
ax.set_title('Effect of coherence on drift rate\nnxx1, seed=42, gain=1.0')
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print(f"v_Intercept: {v_intercept:.3f}")
print(f"v_coherence: {v_coherence:.3f}")

## Posterior Predictive Check

## Posterior Predictive Check

In [ ]:
# Default PPC — compare observed vs predicted RT distributions
if 'posterior_predictive' in idata:
    pp = idata['posterior_predictive'].to_dataset()
    obs = idata['observed_data'].to_dataset()
    
    # Get predicted and observed RTs
    pred_rt = pp['rt,response'].values[..., 0].flatten()  # predicted RTs
    obs_rt = obs['rt,response'].values[:, 0]  # observed RTs
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # RT distribution
    axes[0].hist(obs_rt, bins=50, density=True, alpha=0.6, color='blue', label='observed')
    axes[0].hist(pred_rt, bins=50, density=True, alpha=0.4, color='red', label='predicted')
    axes[0].set_xlabel('RT (s)')
    axes[0].set_ylabel('Density')
    axes[0].set_title('RT Distribution: Observed vs Predicted')
    axes[0].legend()
    
    # Response distribution
    obs_resp = obs['rt,response'].values[:, 1]
    pred_resp = pp['rt,response'].values[..., 1].flatten()
    axes[1].bar(['correct (obs)', 'correct (pred)'], 
                [(obs_resp==1).mean(), (pred_resp==1).mean()],
                color=['blue', 'red'], alpha=0.7)
    axes[1].set_ylabel('Proportion correct')
    axes[1].set_title('Accuracy: Observed vs Predicted')
    
    plt.tight_layout()
    plt.show()
else:
    print("No posterior predictive samples found in idata.")

## Quantile Probability Plot

In [ ]:
# Quantile probability plot
if 'posterior_predictive' in idata:
    obs = idata['observed_data'].to_dataset()
    pp = idata['posterior_predictive'].to_dataset()
    
    obs_rt = obs['rt,response'].values[:, 0]
    obs_resp = obs['rt,response'].values[:, 1]
    pred_rt_all = pp['rt,response'].values[..., 0]
    pred_resp_all = pp['rt,response'].values[..., 1]
    
    quantiles = [0.1, 0.3, 0.5, 0.7, 0.9]
    
    # Observed quantiles split by response
    obs_correct_rt = obs_rt[obs_resp == 1]
    obs_error_rt = obs_rt[obs_resp == -1]
    obs_q_correct = np.quantile(obs_correct_rt, quantiles) if len(obs_correct_rt) > 0 else np.full(len(quantiles), np.nan)
    obs_q_error = np.quantile(obs_error_rt, quantiles) if len(obs_error_rt) > 0 else np.full(len(quantiles), np.nan)
    
    # Predicted quantiles (mean across posterior samples)
    pred_rt_flat = pred_rt_all.reshape(-1, pred_rt_all.shape[-1])
    pred_resp_flat = pred_resp_all.reshape(-1, pred_resp_all.shape[-1])
    
    pred_q_correct = []
    pred_q_error = []
    for i in range(pred_rt_flat.shape[0]):
        rt = pred_rt_flat[i]
        resp = pred_resp_flat[i]
        correct_rt = rt[resp == 1]
        error_rt = rt[resp == -1]
        pred_q_correct.append(np.quantile(correct_rt, quantiles) if len(correct_rt) > 0 else np.full(len(quantiles), np.nan))
        pred_q_error.append(np.quantile(error_rt, quantiles) if len(error_rt) > 0 else np.full(len(quantiles), np.nan))
    
    pred_q_correct = np.nanmean(pred_q_correct, axis=0)
    pred_q_error = np.nanmean(pred_q_error, axis=0)
    
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(obs_q_correct, pred_q_correct, 'bo-', label='correct')
    ax.plot(obs_q_error, pred_q_error, 'rs-', label='error')
    ax.plot([0, max(obs_rt)], [0, max(obs_rt)], 'k--', alpha=0.4, label='identity')
    for i, q in enumerate(quantiles):
        ax.annotate(f'{int(q*100)}%', (obs_q_correct[i], pred_q_correct[i]), 
                   textcoords='offset points', xytext=(5,5), fontsize=8)
    ax.set_xlabel('Observed RT quantiles (s)')
    ax.set_ylabel('Predicted RT quantiles (s)')
    ax.set_title('Quantile Probability Plot\nnxx1, seed=42, gain=1.0 (fixed bound)')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No posterior predictive samples found in idata.")